# Baseline Sweep Analysis

Reads every `sweep_*_folds.tsv` produced by `analysis/sweep.py --folds-root ...` and
answers the four questions the sweep exists to answer:

1. **Which configuration wins each model** — on fold MEAN, never on the test set.
2. **Is the win real or regime-specific** — by comparing the improvement against the
   FOLD SPREAD. A point whose spread exceeds its improvement won on one period.
3. **What each axis actually does** — one curve per axis, so the shape is visible
   rather than just the argmax.
4. **What to hand to `final_phase_test`** — one champion per model, recorded before
   the test set is opened.

### Units, stated once

| column | what one number means | denominator |
|---|---|---|
| `score` | official competition score, averaged over folds | summed over 134 turbines, mean over folds |
| `fold_std` | spread of that score across F1–F5 | 5 folds |
| `pct_capacity` | `score` as a share of farm rated capacity | 201 MW = 134 × 1.5 MW |

`score` is the same quantity as the published 47.850 MW GRU baseline, so it is directly
comparable to it and to the test numbers.

In [ ]:
import os, sys, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# repo layout: notebooks/ -> repo root
_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(_ROOT) != "windevolve_sdwpf":
    _ROOT = os.path.join(_ROOT, "windevolve_sdwpf")
for sub in ["", "evaluation", "analysis"]:
    p = os.path.join(_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)

SWEEP_DIR = os.path.join(_ROOT, "runs", "sweeps")
FIG_DIR = os.path.join(_ROOT, "runs", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

NAVY, GOLD = "#182B49", "#FFCD00"
COLOURS = {"climatology": "#ff7f0e", "arma": "#2ca02c", "gbdt": "#1f77b4",
           "gru_global": "#9467bd", "mlp": "#e377c2", "lstm": "#7f7f7f",
           "transformer": "#bcbd22"}
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), bbox_inches="tight", dpi=160)

print("sweep dir:", SWEEP_DIR)
print("found    :", sorted(os.path.basename(f) for f in glob.glob(os.path.join(SWEEP_DIR, "sweep_*_folds.tsv"))))

## 1 · Load

Only `*_folds.tsv` files are read. A `sweep_<model>.tsv` without the suffix was measured
on the old single split and is **not comparable** — the filename suffix exists precisely
so the two can never be mixed.

In [ ]:
frames = []
for path in sorted(glob.glob(os.path.join(SWEEP_DIR, "sweep_*_folds.tsv"))):
    d = pd.read_csv(path, sep="\t")
    d["model"] = os.path.basename(path).replace("sweep_", "").replace("_folds.tsv", "")
    frames.append(d)

if not frames:
    raise SystemExit("no *_folds.tsv found -- run the sweep with --folds-root first")

raw = pd.concat(frames, ignore_index=True)
df = raw[raw["valid"] == 1].copy()

for c in ["score", "fold_std", "pct_capacity", "rmse", "mae", "fit_seconds"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# expand the per-fold scores so folds can be inspected individually
def _folds(s):
    try:
        return json.loads(s) if isinstance(s, str) and s.strip() else {}
    except Exception:
        return {}
fold_cols = df["fold_scores"].apply(_folds).apply(pd.Series)
df = pd.concat([df, fold_cols], axis=1)
FOLDS = [c for c in fold_cols.columns]

n_bad = len(raw) - len(df)
print("valid rows: {}   invalid: {}".format(len(df), n_bad))
print("models    :", ", ".join(sorted(df["model"].unique())))
print("folds     :", ", ".join(FOLDS))
if n_bad:
    print("\ninvalid points (these were scored and failed, not skipped):")
    display(raw[raw["valid"] != 1][["model", "axis", "label", "reason"]])

## 2 · Champions

One row per model: the configuration with the best fold mean.

**`margin_vs_2nd`** is how much better the champion is than the runner-up. Read it
against `fold_std`. If the margin is smaller than the spread, the "win" is inside the
noise between regimes and the champion is not meaningfully better than the runner-up —
which is a finding about the axis, not a reason to pick differently.

In [ ]:
rows = []
for m, g in df.groupby("model"):
    g = g.sort_values("score")
    best = g.iloc[0]
    second = g.iloc[1] if len(g) > 1 else None
    rows.append({
        "model": m,
        "champion": best["label"],
        "score_MW": round(best["score"], 3),
        "fold_std": round(best["fold_std"], 3),
        "pct_capacity": best["pct_capacity"],
        "margin_vs_2nd": (round(second["score"] - best["score"], 3)
                          if second is not None else np.nan),
        "n_points": len(g),
        "worst_fold": (round(max(best[f] for f in FOLDS if f in best), 3)
                       if FOLDS else np.nan),
    })
champions = pd.DataFrame(rows).sort_values("score_MW").reset_index(drop=True)

champions["real_win"] = np.where(
    champions["margin_vs_2nd"].isna(), "single point",
    np.where(champions["margin_vs_2nd"] > champions["fold_std"],
             "yes", "NO - inside fold spread"))

print("Champions, chosen on FOLD MEAN only. The test set has not been opened.\n")
display(champions)

## 3 · Every axis, as a curve

The argmax alone hides the shape, and the shape is usually the result. A flat curve means
the axis does not matter; a monotone curve that has not flattened means the range was too
narrow and the optimum is outside it.

Error bars are `fold_std` — the spread across regimes, not a standard error.

In [ ]:
models = sorted(df["model"].unique())
n = len(models)
fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(13, 3.1 * ((n + 1) // 2)))
axes = np.atleast_1d(axes).ravel()

for ax, m in zip(axes, models):
    g = df[df["model"] == m]
    for axis, ga in g.groupby("axis"):
        ga = ga.sort_values("score")
        ax.errorbar(range(len(ga)), ga["score"], yerr=ga["fold_std"],
                    fmt="o-", capsize=3, lw=1.4, ms=4, label=axis)
        for i, (_, r) in enumerate(ga.iterrows()):
            ax.annotate(str(r["label"])[:22], (i, r["score"]), fontsize=6.5,
                        rotation=30, xytext=(0, 7), textcoords="offset points")
    best = g["score"].min()
    ax.axhline(best, color=NAVY, ls="--", lw=1,
               label="best {:.2f} MW".format(best))
    ax.set_title(m, color=NAVY, fontweight="bold")
    ax.set_ylabel("fold-mean score (MW)")
    ax.set_xticks([])
    ax.legend(fontsize=7, ncol=2)

for ax in axes[len(models):]:
    ax.axis("off")
plt.tight_layout(); savefig(fig, "sweep_axes"); plt.show()

## 4 · Is the win real, or regime-specific?

The rule agreed before the sweep ran:

> **A point whose fold spread exceeds its improvement over the reference is a
> regime-specific win, not a real one.**

Points below the diagonal improved by less than they vary across folds. They may still be
the best available choice, but the improvement should not be reported as a finding.

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 5.4))
for m, g in df.groupby("model"):
    ref = g[g["axis"] == "reference"]
    base = ref["score"].iloc[0] if len(ref) else g["score"].max()
    improvement = base - g["score"]
    ax.scatter(g["fold_std"], improvement, s=42, alpha=0.85,
               color=COLOURS.get(m), label=m, edgecolor="white", linewidth=0.6)

lim = max(df["fold_std"].max(), 0.2) * 1.15
ax.plot([0, lim], [0, lim], color=NAVY, ls="--", lw=1.2)
ax.annotate("improvement = fold spread", (lim * 0.55, lim * 0.58), fontsize=8,
            color=NAVY, rotation=32)
ax.fill_between([0, lim], [0, 0], [0, lim], color="#d62728", alpha=0.06)
ax.annotate("below the line:\nregime-specific", (lim * 0.55, lim * 0.12),
            fontsize=8.5, color="#8b1a1a")
ax.set_xlabel("fold spread, fold_std (MW)")
ax.set_ylabel("improvement over that model's reference (MW)")
ax.set_title("Real wins sit ABOVE the diagonal", color=NAVY, fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); savefig(fig, "sweep_spread_vs_gain"); plt.show()

## 5 · Per-fold detail for the champions

If a champion is much worse on one fold than the others, its mean is being carried by the
rest. F1 has the least training data (95 days) and F5 the most (215), so a champion that
degrades monotonically from F5 to F1 is telling you the configuration is data-hungry
rather than regime-sensitive — a different problem with a different fix.

In [ ]:
if FOLDS:
    best_rows = df.loc[df.groupby("model")["score"].idxmin()]
    wide = best_rows.set_index("model")[FOLDS]

    fig, ax = plt.subplots(figsize=(10, 0.55 * len(wide) + 2))
    x = np.arange(len(FOLDS)); w = 0.8 / max(1, len(wide))
    for k, (m, row) in enumerate(wide.iterrows()):
        ax.bar(x + k * w - 0.4 + w / 2, row.values, w,
               color=COLOURS.get(m), label=m)
    ax.set_xticks(x); ax.set_xticklabels(FOLDS)
    ax.set_xlabel("fold (F1 has the least training data, F5 the most)")
    ax.set_ylabel("competition score (MW)")
    ax.set_title("Champion configurations, fold by fold", color=NAVY, fontweight="bold")
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout(); savefig(fig, "sweep_champions_by_fold"); plt.show()

    display(wide.round(3))
else:
    print("no per-fold detail in these TSVs")

## 6 · The XGBoost feature tiers

The one ablation in the suite that is a claim rather than a tuning number.

The six standard tiers are strictly **nested** — `power`(9) ⊂ `base`(12) ⊂ `turbine`(22) ⊂
`farm`(26) ⊂ `neighbour`(29) ⊂ `xy`(31) — so `xy − base` is the cumulative effect of four
feature groups added at once and cannot be attributed to any single one.

**`base+nbr`** is the non-nested tier: `base` plus only the three neighbour columns, 15 in
total. `base+nbr − base` is therefore the first clean measurement of what neighbour
information is worth on its own.

In [ ]:
g = df[(df["model"] == "gbdt") & (df["axis"] == "features")].copy()
if len(g):
    SIZE = {"power": 9, "base": 12, "turbine": 22, "farm": 26,
            "neighbour": 29, "xy": 31, "base+nbr": 15}
    g["tier"] = g["label"].str.extract(r"tier=([\w+]+)")[0]
    g["n_cols"] = g["tier"].map(SIZE)
    base = g[g["tier"] == "base"]["score"]
    base = base.iloc[0] if len(base) else np.nan
    g["vs_base_MW"] = (g["score"] - base).round(3)
    g["nested"] = np.where(g["tier"] == "base+nbr", "NO", "yes")

    print("Negative vs_base_MW means the tier BEAT base.\n")
    display(g[["tier", "n_cols", "nested", "score", "fold_std", "vs_base_MW"]]
            .sort_values("n_cols").reset_index(drop=True))

    nbr = g[g["tier"] == "base+nbr"]
    if len(nbr) and np.isfinite(base):
        d = float(nbr["vs_base_MW"].iloc[0]); sd = float(nbr["fold_std"].iloc[0])
        print("\nbase+nbr vs base: {:+.3f} MW  (fold spread {:.3f})".format(d, sd))
        if abs(d) < sd:
            print("  -> inside the fold spread: neighbour features are NOT")
            print("     demonstrably worth anything on their own.")
        elif d < 0:
            print("  -> neighbour features HELP on their own. The earlier x/y null")
            print("     result tested turbine IDENTITY, not spatial STRUCTURE.")
        else:
            print("  -> neighbour features hurt even alone, so the earlier penalty")
            print("     was not caused solely by the sensor columns.")
else:
    print("no gbdt feature-tier rows yet")

## 7 · The suite, side by side

Every model at its champion configuration, against the two external reference points:
the published GRU baseline (47.850 MW) and the best KDD Cup entry (44.917 MW).

These are **CV numbers**, not test numbers, so they are not directly comparable to the
published figures — the reference lines are there for scale, not for ranking.

In [ ]:
PUBLISHED_GRU = 47.850     # official baseline, competition test set
BEST_TEAM = 44.917         # HIK, competition test set

c = champions.sort_values("score_MW")
fig, ax = plt.subplots(figsize=(9.5, 0.5 * len(c) + 2.2))
y = np.arange(len(c))
ax.barh(y, c["score_MW"], xerr=c["fold_std"], capsize=3,
        color=[COLOURS.get(m, NAVY) for m in c["model"]])
ax.set_yticks(y); ax.set_yticklabels(c["model"])
ax.invert_yaxis()
ax.axvline(PUBLISHED_GRU, color=NAVY, ls="--", lw=1.2)
ax.annotate("published GRU 47.850 (test)", (PUBLISHED_GRU, len(c) - 0.4),
            fontsize=8, color=NAVY, rotation=90, va="bottom", ha="right")
ax.axvline(BEST_TEAM, color="#8b1a1a", ls=":", lw=1.2)
ax.annotate("best KDD entry 44.917 (test)", (BEST_TEAM, len(c) - 0.4),
            fontsize=8, color="#8b1a1a", rotation=90, va="bottom", ha="right")
for i, (_, r) in enumerate(c.iterrows()):
    ax.annotate("{:.2f}".format(r["score_MW"]), (r["score_MW"], i),
                xytext=(5, 0), textcoords="offset points", va="center", fontsize=8.5)
ax.set_xlabel("5-fold CV competition score (MW), error bars = fold spread")
ax.set_title("Baseline suite at champion configurations", color=NAVY, fontweight="bold")
plt.tight_layout(); savefig(fig, "sweep_suite"); plt.show()

## 8 · Freeze the picks

Writes `runs/sweeps/champions.json`. **Run this before touching `final_phase_test`.**

The file is the record that the configuration was chosen on folds alone. Once it exists,
the test set is scored once per model at exactly these settings and never re-run — with
five configurations scored on it you no longer have a held-out estimate, you have a sixth
validation split, and the ρ ≈ −0.20 result already showed where that leads.

Each champion is also the seed program handed to its model-family evolve arm.

In [ ]:
out_path = os.path.join(SWEEP_DIR, "champions.json")

payload = {
    "chosen_on": "5-fold rolling-origin CV mean, folds F1-F5",
    "test_set_opened": False,
    "champions": {},
}
for _, r in champions.iterrows():
    src = df[(df["model"] == r["model"]) & (df["label"] == r["champion"])].iloc[0]
    payload["champions"][r["model"]] = {
        "label": r["champion"],
        "axis": src["axis"],
        "config": src.get("config", ""),
        "cv_score_MW": float(r["score_MW"]),
        "fold_std": float(r["fold_std"]),
        "margin_vs_2nd": (None if pd.isna(r["margin_vs_2nd"])
                          else float(r["margin_vs_2nd"])),
        "real_win": r["real_win"],
        "test_score_MW": None,      # fill in AFTER scoring final_phase_test once
    }

with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)

print("wrote", out_path)
print(json.dumps(payload, indent=2)[:1400])

### Then, and only then

For each model, score the frozen configuration on the test set exactly once:

```bash
python evaluation/evaluator.py baselines/neural/mlp.py --test
```

Paste the result into `test_score_MW` in `champions.json` and set
`"test_set_opened": true`. Each model then has the two numbers it needs — the CV score
for internal comparison and selection, and the test score for external comparison against
the published 47.850 MW.